# SCIDOCS + HyDE (Hypothetical Document Embeddings)

**Context-Enriched Retrieval for RAG — Iris.ai**

HyDE replaces the raw query with an LLM-generated *hypothetical answer passage*, then embeds **that**
and searches with it. The fake passage "looks like" a real corpus document, so it can match better —
even though it may contain wrong facts (it is used **only** for retrieval, never as evidence).

**Two distinct models (don't conflate them):**
- **Generator** = OpenAI `gpt-4o-mini` — writes the hypothetical passage (HyDE-only step).
- **Embedder** = `Octen/Octen-Embedding-0.6B` — same fixed model as the baseline; embeds the passage.

This notebook **reuses the cached corpus embeddings from the baseline run** (no re-embedding), so the
only thing that changes vs. baseline is the *query representation*. It ends with a
**baseline-vs-HyDE comparison**: metric deltas, per-query win/loss, and a Wilcoxon significance test.

## 1. Config & setup

In [1]:
import os, json, time, hashlib, random
from pathlib import Path
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import pytrec_eval
from scipy import stats
from dotenv import load_dotenv
from openai import OpenAI

# ---- Config ---------------------------------------------------------------
MODEL_NAME     = "Octen/Octen-Embedding-0.6B"   # embedder (fixed, same as baseline)
GEN_MODEL      = "gpt-4o-mini"                   # generator (HyDE-only)
DATA_DIR       = Path("../scidocs")
SPLIT          = "test"
BASELINE_DIR   = Path("results/scidocs/baseline")        # we reuse its corpus embeddings
OUT_DIR        = Path("results/scidocs/hyde")
TOP_K          = 1000
BATCH_SIZE     = 32
SEED           = 42
RUN_TAG        = "hyde"

# ---- HyDE knobs (ablation-friendly) --------------------------------------
N_HYPO         = 1            # hypothetical docs generated per query (paper used up to 8)
TEMPERATURE    = 0.7         # generation temperature
MAX_TOKENS     = 220         # length of each hypothetical passage
HYPO_EMBED_AS  = "document"  # embed the fake passage in the corpus (document) space
INCLUDE_QUERY  = False       # if True, average the real query embedding into the HyDE vector

OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
load_dotenv()                       # reads OPENAI_API_KEY from .env (never printed)
client = OpenAI(max_retries=8)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — create a .env file with it."
print("device   :", DEVICE)
print("generator:", GEN_MODEL, "| embedder:", MODEL_NAME)

/Users/katerina_dimitrova/Downloads/Iris.ai RAG research/HyDE context Enrichment test/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device   : mps
generator: gpt-4o-mini | embedder: Octen/Octen-Embedding-0.6B


## 2. Load data + reuse the baseline's cached corpus embeddings

The corpus matrix is loaded straight from `results/scidocs/baseline/` — identical document vectors, so the
comparison isolates the query-side change. Re-run the baseline notebook first if these files are missing.

In [2]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def load_qrels(path):
    qrels = {}
    with open(path, encoding="utf-8") as f:
        next(f)
        for line in f:
            qid, did, score = line.rstrip("\n").split("\t")
            if int(score) > 0:
                qrels.setdefault(qid, {})[did] = int(score)
    return qrels

# queries + qrels (same test split as baseline)
qrels = load_qrels(DATA_DIR / "qrels" / f"{SPLIT}.tsv")
all_queries = {r["_id"]: r["text"] for r in load_jsonl(DATA_DIR / "queries.jsonl")}
queries = {qid: all_queries[qid] for qid in qrels if qid in all_queries}
query_ids = list(queries.keys())

# reuse cached corpus embeddings from the baseline run
assert (BASELINE_DIR / "corpus_emb.npy").exists(), "Run baseline_nfcorpus.ipynb first (need cached corpus_emb)."
corpus_emb = np.load(BASELINE_DIR / "corpus_emb.npy")
corpus_ids = json.loads((BASELINE_DIR / "corpus_ids.json").read_text())
print(f"queries: {len(queries)} | corpus_emb: {corpus_emb.shape} (reused from baseline)")

queries: 323 | corpus_emb: (3633, 1024) (reused from baseline)


## 3. Generate hypothetical documents (OpenAI)

Cached to `results/scidocs/hyde/hyde_docs.json` per query — re-runs skip queries already generated, so you
never pay twice. Delete that file to regenerate from scratch.

In [3]:
PROMPT_TEMPLATE = (
    "Write a short scientific passage, in the style of a scientific paper abstract, that could "
    "directly relate to the following scientific paper query. Be specific and factual in tone; "
    "around 80-120 words. Do not add a title or any preamble.\n\n"
    "Question: {question}\nPassage:"
)

def generate_one(question, retries=4):
    msg = [{"role": "user", "content": PROMPT_TEMPLATE.format(question=question)}]
    for attempt in range(retries):
        try:
            r = client.chat.completions.create(
                model=GEN_MODEL, messages=msg,
                temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
            return r.choices[0].message.content.strip()
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(2 * (attempt + 1))

HYPO_PATH = OUT_DIR / "hyde_docs.json"
hypo = json.loads(HYPO_PATH.read_text()) if HYPO_PATH.exists() else {}
todo = [q for q in query_ids if q not in hypo or len(hypo[q]) < N_HYPO]
print(f"to generate: {len(todo)} queries x {N_HYPO} docs  (cached: {len(query_ids)-len(todo)})")

for qid in tqdm(todo):
    docs = hypo.get(qid, [])
    while len(docs) < N_HYPO:
        docs.append(generate_one(queries[qid]))
    hypo[qid] = docs
    HYPO_PATH.write_text(json.dumps(hypo))   # incremental save (crash-safe)
print("done. example generation:\n", hypo[query_ids[0]][0][:400], "...")

to generate: 323 queries x 1 docs  (cached: 0)


100%|██████████| 323/323 [12:49<00:00,  2.38s/it]

done. example generation:
 Recent epidemiological studies have investigated the potential link between statin use and breast cancer incidence. A meta-analysis of 15 cohort studies involving over 1.5 million women found no significant association between statin therapy and the risk of developing breast cancer (relative risk: 0.95; 95% CI: 0.87-1.04). Additionally, a large randomized controlled trial demonstrated that long-te ...


## 4. Embed hypothetical docs with Octen → build HyDE query vectors

Each query's hypothetical passage(s) are embedded and mean-pooled into a single search vector
(optionally averaged with the real query embedding). Same embedder, normalization, and pooling as baseline.

In [4]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = 512

def embed(texts, prompt_name):
    return model.encode(texts, prompt_name=prompt_name, batch_size=BATCH_SIZE,
                        normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=True).astype(np.float32)

# flatten -> embed all hypothetical docs at once -> regroup -> mean-pool per query
flat_texts, owner = [], []
for qid in query_ids:
    for d in hypo[qid][:N_HYPO]:
        flat_texts.append(d); owner.append(qid)
hypo_emb = embed(flat_texts, prompt_name=HYPO_EMBED_AS)

by_q = {q: [] for q in query_ids}
for v, qid in zip(hypo_emb, owner):
    by_q[qid].append(v)
hyde_vecs = np.vstack([np.mean(by_q[q], axis=0) for q in query_ids]).astype(np.float32)

if INCLUDE_QUERY:
    q_emb = embed([queries[q] for q in query_ids], prompt_name="query")
    hyde_vecs = (hyde_vecs + q_emb) / 2.0

# renormalize so cosine == dot product
hyde_vecs /= np.linalg.norm(hyde_vecs, axis=1, keepdims=True) + 1e-12
print("HyDE query vectors:", hyde_vecs.shape)

Batches: 100%|██████████| 11/11 [00:16<00:00,  1.47s/it]

HyDE query vectors: (323, 1024)


## 5. Retrieve + evaluate (same metrics as baseline)

In [5]:
sims = hyde_vecs @ corpus_emb.T
k = min(TOP_K, sims.shape[1])
topk_idx = np.argpartition(-sims, k - 1, axis=1)[:, :k]
run = {}
for i, qid in enumerate(query_ids):
    idx = topk_idx[i]
    order = idx[np.argsort(-sims[i, idx])]
    run[qid] = {corpus_ids[j]: float(sims[i, j]) for j in order}

measures = {"ndcg_cut.1,3,5,10", "recall.3,5,10,20,100", "map", "P.10", "recip_rank", "success.1,5,10"}
per_query = pytrec_eval.RelevanceEvaluator(qrels, measures).evaluate(run)
avg = lambda m: float(np.mean([per_query[q][m] for q in per_query]))
metrics = {
    "NDCG@1": avg("ndcg_cut_1"), "NDCG@3": avg("ndcg_cut_3"), "NDCG@5": avg("ndcg_cut_5"), "NDCG@10": avg("ndcg_cut_10"),
    "Recall@3": avg("recall_3"), "Recall@5": avg("recall_5"), "Recall@10": avg("recall_10"),
    "Recall@20": avg("recall_20"), "Recall@100": avg("recall_100"),
    "Hit@1": avg("success_1"), "Hit@5": avg("success_5"), "Hit@10": avg("success_10"),
    "MAP": avg("map"), "P@10": avg("P_10"), "MRR": avg("recip_rank"),
}
hyde_pq_ndcg10 = {q: per_query[q]["ndcg_cut_10"] for q in per_query}

HEADLINE = {"NDCG@10", "Recall@5", "Recall@10", "Hit@5"}
print(f"{'metric':<12} {'score':>8}")
print("-" * 26)
for m, v in metrics.items():
    print(f"{m:<12} {v:>8.4f}{'  <-- headline' if m in HEADLINE else ''}")

metric          score
--------------------------
NDCG@1         0.4520
NDCG@3         0.4018
NDCG@5         0.3835
NDCG@10        0.3559  <-- headline
Recall@3       0.1041
Recall@5       0.1378  <-- headline
Recall@10      0.1780  <-- headline
Recall@20      0.2172
Recall@100     0.3448
Hit@1          0.4737
Hit@5          0.6687  <-- headline
Hit@10         0.7245
MAP            0.1825
P@10           0.2653
MRR            0.5606


## 6. Save HyDE outputs

In [6]:
with open(OUT_DIR / "run.tsv", "w") as f:
    for qid in query_ids:
        for rank, (did, score) in enumerate(run[qid].items(), start=1):
            f.write(f"{qid}\tQ0\t{did}\t{rank}\t{score:.6f}\t{RUN_TAG}\n")
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(OUT_DIR / "per_query_ndcg10.json").write_text(json.dumps(hyde_pq_ndcg10, indent=2))
config = {
    "run_tag": RUN_TAG, "enrichment": "HyDE", "model_name": MODEL_NAME, "generator": GEN_MODEL,
    "n_hypo": N_HYPO, "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS,
    "hypo_embed_as": HYPO_EMBED_AS, "include_query": INCLUDE_QUERY,
    "dataset": "SCIDOCS", "split": SPLIT, "n_queries": len(queries),
    "top_k": k, "seed": SEED, "device": DEVICE,
    "prompt_template": PROMPT_TEMPLATE, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))
print("saved to", OUT_DIR.resolve())
for p in sorted(OUT_DIR.iterdir()):
    print("  ", p.name)

saved to /Users/katerina_dimitrova/Downloads/Iris.ai RAG research/HyDE context Enrichment test/results/hyde
   config.json
   hyde_docs.json
   metrics.json
   per_query_ndcg10.json
   run.tsv


## 7. Baseline vs HyDE — deltas, win/loss, significance

The core of the comparison. Averages can hide that a method helps some queries and hurts others, so
we also count per-query wins/losses on NDCG@10 and run a paired **Wilcoxon signed-rank** test to check
the difference is not just noise.

In [7]:
base_metrics = json.loads((BASELINE_DIR / "metrics.json").read_text())
base_pq = json.loads((BASELINE_DIR / "per_query_ndcg10.json").read_text())

# ---- side-by-side metric table -------------------------------------------
print(f"{'metric':<12} {'baseline':>9} {'HyDE':>9} {'delta':>9}")
print("-" * 42)
for m in metrics:
    b, h = base_metrics.get(m, float('nan')), metrics[m]
    d = h - b
    flag = "  ^" if d > 0 else ("  v" if d < 0 else "")
    print(f"{m:<12} {b:>9.4f} {h:>9.4f} {d:>+9.4f}{flag}")

# ---- per-query win/loss/tie on NDCG@10 -----------------------------------
qs = [q for q in query_ids if q in base_pq]
b = np.array([base_pq[q] for q in qs])
h = np.array([hyde_pq_ndcg10[q] for q in qs])
diff = h - b
wins, losses, ties = int((diff > 1e-9).sum()), int((diff < -1e-9).sum()), int((np.abs(diff) <= 1e-9).sum())
print(f"\nNDCG@10 per-query (n={len(qs)}):  wins={wins}  losses={losses}  ties={ties}")
print(f"mean delta = {diff.mean():+.4f} | rescued (0 -> >0): "
      f"{int(((b==0)&(h>0)).sum())} | lost (>0 -> 0): {int(((b>0)&(h==0)).sum())}")

# ---- Wilcoxon signed-rank (paired, non-parametric) -----------------------
nz = diff[np.abs(diff) > 1e-9]
if len(nz) > 0:
    stat, p = stats.wilcoxon(nz)
    verdict = "SIGNIFICANT" if p < 0.05 else "not significant"
    print(f"Wilcoxon signed-rank: stat={stat:.1f}, p={p:.4g}  ->  {verdict} at alpha=0.05")
else:
    print("Wilcoxon: no non-zero differences to test.")

# ---- where did HyDE help / hurt the most? --------------------------------
order = np.argsort(diff)
print("\nBiggest HyDE WINS:")
for i in order[::-1][:5]:
    print(f"  {diff[i]:+.3f}  {queries[qs[i]][:70]}")
print("Biggest HyDE LOSSES:")
for i in order[:5]:
    print(f"  {diff[i]:+.3f}  {queries[qs[i]][:70]}")

metric        baseline      HyDE     delta
------------------------------------------
NDCG@1          0.4737    0.4520   -0.0217  v
NDCG@3          0.4206    0.4018   -0.0187  v
NDCG@5          0.4002    0.3835   -0.0167  v
NDCG@10         0.3654    0.3559   -0.0095  v
Recall@3        0.1112    0.1041   -0.0071  v
Recall@5        0.1402    0.1378   -0.0024  v
Recall@10       0.1747    0.1780   +0.0033  ^
Recall@20       0.2167    0.2172   +0.0006  ^
Recall@100      0.3424    0.3448   +0.0024  ^
Hit@1           0.4923    0.4737   -0.0186  v
Hit@5           0.6842    0.6687   -0.0155  v
Hit@10          0.7245    0.7245   +0.0000
MAP             0.1911    0.1825   -0.0086  v
P@10            0.2687    0.2653   -0.0034  v
MRR             0.5851    0.5606   -0.0245  v

NDCG@10 per-query (n=323):  wins=103  losses=110  ties=110
mean delta = -0.0095 | rescued (0 -> >0): 15 | lost (>0 -> 0): 15
Wilcoxon signed-rank: stat=10518.5, p=0.3301  ->  not significant at alpha=0.05

Biggest HyDE WINS:
 